In [ ]:
import torch
from torch import nn
from torch import Tensor
import numpy as np
import matplotlib.pyplot as plt

import math
from dataclasses import dataclass
from torchvision.datasets import CIFAR10
from torchvision import transforms

from torch.utils.data import DataLoader, random_split



class MultiHeadAttention(nn.Module):
    """Models a mutli-head self-attention block for an Encoder"""

    def __init__(self, n_heads: int, embd_size: int) -> None:
        """Initializes the mult-head self-attention block.
        
        Args:
            n_heads: Number of attention heads in the attention head of each encoder.
            embd_size: Dimension size of the patch embeddings.
        """
        super().__init__()

        self.n_heads = n_heads
        self.embd_size = embd_size

        assert self.embd_size % self.n_heads == 0
        self.head_embd_size = self.embd_size // self.n_heads

        self.QKV = nn.Linear(self.embd_size, self.n_heads * 3*self.head_embd_size)

        # Linearly combines the output from each attention head
        self.W_O = nn.Linear(self.embd_size, self.embd_size)


    def forward(self, x: Tensor) -> Tensor:
        """Computes mutli-head self-attention over the input token sequence.
        
        Args:
            x: A tensor of shape (T, embd_size)

        Returns:
            A tensor of shape (T, embd_size) containing the contextualized representation of each input token.
        """
        T = x.shape[0]

        qkv = self.QKV(x).reshape(T, self.n_heads, -1).transpose(1, 0) # (n_heads, T, 3*head_embd_size)
        
        Q, K, V = qkv.split(self.head_embd_size, dim = -1) # (n_heads, T, head_embd_size)

        # (n_heads, T, T)
        attn_scores = torch.softmax((Q @ K.transpose(2, 1)) / math.sqrt(self.head_embd_size), dim = -1)
        z = attn_scores @ V # (n_heads, T, head_embd_size)
        z = z.transpose(1, 0).reshape(T, -1) # (T, n_heads * head_embd_size) = (T, embd_size)
        z = self.W_O(z) # (T, embd_size)

        return z


class MLP(nn.Module):
    """Models a simple Multi-Layer Perceptron"""

    
    def __init__(self, embd_size: int, hidden_size: int, output_size: int = None) -> None:
        """Initializes a MLP
        Args:
            hidden_size: Number of hidden neurons
            output_size: Number of output neurons (if applicable)
        """
        
        super().__init__()
        self.layer1 = nn.Linear(embd_size, hidden_size)
        self.gelu = nn.GELU()

        if output_size is not None:
            self.layer2 = nn.Linear(hidden_size, output_size)
        else: 
            self.layer2 = nn.Linear(hidden_size, embd_size)

    
    def forward(self, x: Tensor) -> Tensor:
        """Maps input tokens through a two-layer MLP.
        
        Applies a linear transformation, GELU activation, and a final
        linear transformation to each input token independently.
    
        Args:
            x: Tensor of shape (T, embd_size)

        Returns:
            Tensor of shape (T, output_size) if output_size was specified,
            otherwise (T, embd_size).
        """
        
        return self.layer2(self.gelu(self.layer1(x)))


class Classifier(nn.Module):
    """Classifies image embeddings using an MLP or a fine-tuned linear layer."""

    def __init__(self, embd_size: int, hidden_size: int, output_size: int) -> None:
        """Initializes a Classifier

        Args:
            embd_size: Dimension of the image embedding
            output_size: Number of output neurons; the number of image classes
            hidden_size: Number of hidden neurons in the MLP
        """
        super().__init__()
        self.classifer = MLP(embd_size, hidden_size, output_size)
        self.classifier_fine_tune = nn.Linear(embd_size, output_size)

    
    def forward(self, x: Tensor, fine_tune: bool) -> Tensor:
        """Maps input image embedding to the logits for the image classes

        Args:
            x: Tensor of shape embd_size
            fine_tune: If true, classifies the image embedding with the fine-tuned layer,
            otherwise uses the MLP.
        
        Returns: 
            Tensor of shape output_size
        """
        
        if fine_tune: 
            return self.classifier_fine_tune(x)
        return self.classifer(x)
    

class EncoderBlock(nn.Module):
    """Models an Encoder block in the Vision Transformer."""

    def __init__(self, n_heads: int, embd_size: int) -> None:
        """Initializes an Encoder block.
        
        Args:
            n_heads: Number of heads in the self-attention block
            embd_size: Dimension size of the patch embeddings
        """
        super().__init__()
        self.ln = nn.LayerNorm(embd_size)
        self.msa = MultiHeadAttention(n_heads, embd_size)
        self.mlp = MLP(embd_size, 4*embd_size)


    def forward(self, x: Tensor) -> Tensor:
        """Applies layer normalization and multi-head self-attention to the input.

        Adds the original input to the attention output through a residual
        connection.

        Args:
            x: Input tensor of shape (T, embd_size).

        Returns:
            A tensor of shape (T, embd_size), containing the attention output 
            with the residual connection applied.
        """
        return self.msa(self.ln(x)) + x

@dataclass
class ViTConfig:
    """Stores configuration parameters for a ViT.
    
    Atrributes:
        n_heads: Number of attention heads in the attention head of each encoder.
        embd_size: Dimension size of the patch embeddings.
        patch_size: Length of each image patch.
        config.: Number of encoders in the ViT.
        input_shape: Tuple containing the height, width, and channels of the input images.
        output_size: Number of image classes.
    """
    n_heads: int = 4
    embd_size: int = 128
    patch_size: int = 2
    num_encoders: int = 4
    input_shape: tuple = (32, 32, 3)
    output_size: int = 10


class ViT(nn.Module):
    """Models a Vision Transformer according to the original "An Image is Worth 16x16 Words" paper.""" 

    def __init__(self, config: ViTConfig) -> None:
        """Initializes a Vision Transformer.

        Args:
            config: The ViTConfig that will be used to initialize the Vision Transformer
        """
        super().__init__()
        self.config = config

        self.patch_size = config.patch_size
        H, W, C = config.input_shape

        assert H % config.patch_size == 0 and W % config.patch_size == 0, f"Invalid patch size {config.patch_size} for {H}x{W} image."
        num_patches = H*W // config.patch_size**2


        self.E = nn.Linear(config.patch_size**2 * C, config.embd_size, bias=False)
        self.E_pos = nn.Parameter(torch.randn(size = (num_patches+1, config.embd_size)))
        self.CLS = nn.Parameter(torch.randn(size = (config.embd_size,)))

        self.encoders = nn.ModuleList(
            config.num_encoders * [EncoderBlock(config.n_heads, config.embd_size)]
        )

        self.classifier = Classifier(config.embd_size, 4*config.embd_size, config.output_size)


    def forward(self, x: Tensor, fine_tune: bool = False) -> Tensor:
        """Computes class logits from input images
        
        Args:
            x: Tensor of shape (C, H, W) containing the input images, where C is the 
            number of channels, H is the height of the image, and W is the width of the image.
            fine_tune: Whether the function is being called during fine-tuning

        Returns:
            A tensor of shape output_size containing the logits for the input image, where output_size 
            is the number of image classes.
        """

        H, W, C = x.shape

        image_patches = torch.stack([
            x[i:i+self.patch_size, j:j+self.patch_size, :].reshape(-1)
            for j in range(0, W, self.patch_size) 
            for i in range(0, H, self.patch_size)
        ]) # (num_patches, P*P*C)

        patch_embeddings = self.E(image_patches) # (num_patches, embd_size)
        patch_embeddings = torch.cat(
            [self.CLS.unsqueeze(0), patch_embeddings],
            dim = 0
        ) # (num_patches+1, embd_size)

        z = patch_embeddings + self.E_pos
        
        for encoder in self.encoders:
            z = encoder(z)

        return self.classifier(z[0], fine_tune)












In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x : x.permute(1, 2, 0)) # (C, H, W) -> (H, W, C)
])


train_dataset = CIFAR10(root="data", train=True, transform=transform, download=True)
test_dataset = CIFAR10(root="data", train=False, transform=transform, download=True)

torch.manual_seed(42)
train_dataset, val_dataset = random_split(train_dataset, [40000, 10000])

batch_size = 128

train_dataloader = DataLoader(
    train_dataset,
    batch_size = batch_size,
    shuffle = True
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size = batch_size,
    shuffle = False
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size = batch_size,
    shuffle = False
)

config = ViTConfig()
model = ViT(config)

lr = 1e-3
weight_decay = 1e-4

optimizer = torch.optim.AdamW(params=model.parameters(), lr=lr, weight_decay=weight_decay)
loss_fn = torch.nn.CrossEntropyLoss()

total_loss = 0.0
model.train()
for i, train_batch in enumerate(train_dataloader):
    images, labels = train_batch

    optimizer.zero_grad()

    logits = model(images[0])
    loss = loss_fn(logits, labels[0])

    loss.backward()
    optimizer.step()    

    break
